In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Shared telescope geometry, the danish-based donut simulator, and figure
# helpers live in the installed `shape_vs_intensity` package (pip install -e .).
import galsim.zernike as gzk
from shape_vs_intensity import config as C
from shape_vs_intensity import sim
from shape_vs_intensity.plotutils import use_style

use_style()

In [ ]:
DEFOCAL_TYPE = "intra"

In [ ]:
DIAGRAM_L = 1.25

def simulate_donut(ax, zernikes, defocal_type="intra"):
    """Simulate a donut image with the given aberrations

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes on which to draw the circles.
    zernikes : dict
        Aberration Zernike amplitudes, as ``{noll_index: coefficient_in_meters}``
    defocal_type : str
        "intra" or "extra".
    """
    img = sim.simulate_donut(zernikes=zernikes, defocal_type=defocal_type)

    crop = C.NPIX // 2 - int(round(C.R_DONUT_PIX * DIAGRAM_L))
    ax.imshow(img[crop:-crop, crop:-crop], origin="lower")
    ax.set(xticks=[], yticks=[])

def map_circles(ax, zernikes, defocal_type="intra", N=5):
    """Map concentric pupil circles through a displacement field.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes on which to draw the circles.
    zernikes : dict
        Aberration Zernike amplitudes, as ``{noll_index: coefficient_in_meters}``
    defocal_type : str
        "intra" or "extra".
    N : int, optional
        Number of pupil circles to draw.
    """
    displacement = zernike_displacement(zernikes, defocal_type=defocal_type)

    theta = np.linspace(0, 2 * np.pi, 10_000)
    for r in np.linspace(C.EPS, 1, N):
        rho = r * np.ones_like(theta)
        x0 = rho * np.cos(theta)
        y0 = rho * np.sin(theta)
        dx, dy = displacement(rho, theta)
        ax.plot(x0 + dx, y0 + dy, c="C1", lw=0.5)

    ax.set(
        xlim=(-DIAGRAM_L, +DIAGRAM_L),
        ylim=(-DIAGRAM_L, +DIAGRAM_L),
        aspect="equal",
        xticks=[],
        yticks=[],
    )


# Calibrate ring displacements to wavefront coefficients (meters).  A defocused
# image maps pupil rays by dp = -grad(W); adding a coefficient c of mode j to the
# intra-focal donut therefore moves a pupil ring by
#   dp = -c * grad(Z_j) / (a4 * dZ4/drho|_1)   [in donut-radius units],
# where a4 = C.DEFOCUS_Z4 is the reference defocus that sets the donut radius
# (the ray-scale constant cancels in the ratio).  Baking this factor into
# zernike_displacement means the SAME numeric coefficient drives both the
# schematic (map_circles) and the donut (sim.simulate_donut(zernikes={j: c})).
_Z4_EDGE_SLOPE = float(
    gzk.Zernike(np.array([0.0, 0.0, 0.0, 0.0, 1.0]), R_outer=1.0, R_inner=C.EPS)
    .gradX(1.0, 0.0)
)
RING_SCALE = 1.0 / (C.DEFOCUS_Z4 * _Z4_EDGE_SLOPE)


def zernike_displacement(zernikes, defocal_type="intra"):
    """Return ring displacement for the given aberrations.

    Parameters
    ----------
    zernikes : dict
        Aberration Zernike amplitudes, as ``{noll_index: coefficient_in_meters}``
    defocal_type : str
        "intra" or "extra".

    Returns
    -------
    callable
        ``displacement(rho, theta) -> (dx, dy)`` in donut-radius units per meter
        of Zernike coefficient.
    """
    coeff = np.zeros(max(zernikes.keys()) + 1)
    for j, amp in zernikes.items():
        coeff[j] = amp
    zern = gzk.Zernike(coeff, R_outer=1.0, R_inner=C.EPS)
    grad_x, grad_y = zern.gradX, zern.gradY

    if defocal_type == "intra":
        scale = C.INTRA_DEFOCUS_SIGN * RING_SCALE
    elif defocal_type == "extra":
        scale = C.EXTRA_DEFOCUS_SIGN * RING_SCALE

    def displacement(rho, theta):
        x = rho * np.cos(theta)
        y = rho * np.sin(theta)
        return scale * grad_x(x, y), scale * grad_y(x, y)

    return displacement

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("Defocus\n$\\nu = 0,\\, m=0$")
axes[0, 1].set_title("Tilt\n$\\nu = 0,\\, m=1$")

for col, zernikes in enumerate([{4: 0.8e-5}, {2: 3e-5}]):
    map_circles(axes[0, col], zernikes, DEFOCAL_TYPE)
    simulate_donut(axes[1, col], zernikes, DEFOCAL_TYPE)

fig.savefig(C.FIGDIR / "aberrations_defocus_tilt.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8), constrained_layout=True, dpi=120)

axes[0, 0].set_title("Astigmatism\n$\\nu = 0,\\, m=2$")
axes[0, 1].set_title("Trefoil\n$\\nu = 0,\\, m=3$")
axes[0, 2].set_title("Quadrafoil\n$\\nu = 0,\\, m=4$")
axes[0, 3].set_title("Pentafoil\n$\\nu = 0,\\, m=5$")

for col, zernikes in enumerate([{6: 7e-6}, {10: 2e-6}, {14: 1e-6}, {20: 1e-6}]):
    map_circles(axes[0, col], zernikes, DEFOCAL_TYPE)
    simulate_donut(axes[1, col], zernikes, DEFOCAL_TYPE)

fig.savefig(C.FIGDIR / "aberrations_astig_mfoil.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("Spherical\n$\\nu = 1,\\, m=0$")
axes[0, 1].set_title("Coma\n$\\nu = 1,\\, m=1$")

for col, zernikes in enumerate([{11: 3e-7}, {8: 2e-6}]):
    map_circles(axes[0, col], zernikes, DEFOCAL_TYPE)
    simulate_donut(axes[1, col], zernikes, DEFOCAL_TYPE)

fig.savefig(C.FIGDIR / "aberrations_spherical_coma.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8), constrained_layout=True, dpi=120)

axes[0, 0].set_title("2nd Astigmatism\n$\\nu = 1,\\, m=2$")
axes[0, 1].set_title("2nd Trefoil\n$\\nu = 1,\\, m=3$")
axes[0, 2].set_title("2nd Quadrafoil\n$\\nu = 1,\\, m=4$")
axes[0, 3].set_title("2nd Pentafoil\n$\\nu = 1,\\, m=5$")

for col, zernikes in enumerate([{12: 0.75e-6}, {18: 0.4e-6}, {26: 0.22e-6}, {34: 0.12e-6}]):
    map_circles(axes[0, col], zernikes, DEFOCAL_TYPE)
    simulate_donut(axes[1, col], zernikes, DEFOCAL_TYPE)

fig.savefig(C.FIGDIR / "aberrations_2nd_astig_mfoil.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("2nd Spherical\n$\\nu = 2,\\, m=0$")
axes[0, 1].set_title("2nd Coma\n$\\nu = 2,\\, m=1$")

for col, zernikes in enumerate([{22: 0.10e-6}, {16: 0.2e-6}]):
    map_circles(axes[0, col], zernikes, DEFOCAL_TYPE)
    simulate_donut(axes[1, col], zernikes, DEFOCAL_TYPE)

fig.savefig(C.FIGDIR / "aberrations_2nd_spherical_coma.pdf", bbox_inches="tight")